# A9 on an open-weights model (Qwen2.5 7B) on a free Colab GPU

Pre-registration: `experiments/PREREGISTRATION_A9.md`, amendment 2. Before starting: **Runtime → Change runtime type → T4 GPU → Save**. Then run the cells in order (Shift + Enter). Keep this tab open.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Install Ollama and start it (about 1-2 minutes)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import os, subprocess, time
env = dict(os.environ, OLLAMA_NUM_PARALLEL='4', OLLAMA_KEEP_ALIVE='24h')
srv = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:7b
!ollama list

In [ ]:
# 2. Get the sealed code and run the offline tests (0 model calls)
!git clone -q https://github.com/r-a-elu-architecture/RSOS-Language.git /content/RSOS-Language
%cd /content/RSOS-Language
!git log --oneline -3
!python experiments/test_a9.py | tail -1

In [ ]:
# 3. Smoke: one task per arm on seed 999 (never scored)
%env A7_BACKEND=ollama
%env A7_LOCAL_MODEL=qwen2.5:7b
%env A7_LABEL=colab_qwen25_7b
%env A7_WORKERS=4
!python experiments/a9_run.py smoke X5b,CAb,B2,CAv

In [ ]:
# 4. AFTER 10:30 (UTC+02:00) Helsinki, Kyiv, Riga, Sofia, Tallinn, Vilnius (08:30 UTC): fetch the beacon and run every cell (resumable: run again if it stops)
!python experiments/a9_beacon.py
!python experiments/a9_run.py all

In [ ]:
# 5. Download the results (run any time; it packs whatever is finished)
!cd /content/RSOS-Language && zip -q -r /content/a9_colab_results.zip results/a9_* && ls -la /content/a9_colab_results.zip
from google.colab import files
files.download('/content/a9_colab_results.zip')